In [1]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.9 MB/s eta 0:00:00:00:01


In [4]:
%%writefile train_ocr.py

import os
import time
import random
import json
import math
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
import torch.distributed as dist
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

from datasets import load_dataset
from datasets import Image as HFImage
import io
import glob
import shutil
import fsspec
from transformers import (
    VisionEncoderDecoderModel,
    AutoImageProcessor,
    AutoTokenizer,
    SwinModel,
    BartForCausalLM,
    GenerationConfig,
    get_cosine_schedule_with_warmup,
)
from jiwer import cer, wer
from tqdm import tqdm

SEED = 42

MAX_SAMPLES_PER_DATASET = None
MJSYNTH_CAP = 50000
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1

# Oversampling factors applied per-source AFTER capping, to correct for
# MJSynth dominating the raw example count. Each source's example list is
# repeated `factor` times (duplicated raw items still get a fresh random
# augmentation draw each epoch, so this is not simply "the exact same image
# N times" from the model's point of view during training).
OVERSAMPLE_FACTORS = {
    "SROIE_2019_text_recognition": 1,
    "IAM-line": 3,
    "IIIT5K_OCR": 2,
    "ICDAR2019-SROIE": 8,
    "MJSynth_text_recognition (capped, newworld)": 1,
}

BATCH_SIZE = 6                    # per-step (micro-batch) size -- lowered from 12 to fit GPU memory
GRAD_ACCUM_STEPS = 2              # accumulate 2 micro-batches -> effective batch size stays 12,
                                   # preserving the LR/weight-decay tuning already done around batch=12
LEARNING_RATE = 2e-5             # LR for pretrained encoder/decoder self-attention params
CROSS_ATTN_LEARNING_RATE = 5e-5  # LR for randomly-initialized cross-attention params
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05               # fraction of total steps used for linear->cosine warmup
MAX_GRAD_NORM = 1.0
# EPOCHS is the ABSOLUTE total across both training phases: the previous run
# already completed WARM_START_EPOCH_OFFSET epochs (with the old, less
# regularized hyperparameters), so this continuation phase runs epochs
# WARM_START_EPOCH_OFFSET+1 .. EPOCHS (i.e. displayed as "epoch 21" through
# "epoch 30"), not epoch 1 through 10 again.
EPOCHS = 30
WARM_START_EPOCH_OFFSET = 20  # epochs already completed in the previous (pre-regularization) run

LABEL_SMOOTHING = 0.1
DROPOUT_P = 0.15  # applied to every nn.Dropout layer in the model after loading

# Augmentation ramping: probability of each augmentation op increases linearly
# from AUGMENT_P_START (epoch 0) to AUGMENT_P_END (final epoch). This lets the
# model learn clean structure fast early on, then leans on augmentation more
# for regularization once it has a stable baseline.
AUGMENT_P_START = 0.15
AUGMENT_P_END = 0.45

NUM_WORKERS = 2
BEST_MODEL_DIR = "./best_ocr_model_v2"
FINAL_MODEL_DIR = "./final_ocr_model_v2"


# ---------------------------------------------------------------------------
# Checkpoint paths for THIS (regularized) run. Kept separate from the old
# run's checkpoint dir so we never overwrite the previous best/final model
# before confirming the new one is actually better.
# ---------------------------------------------------------------------------
CHECKPOINT_DIR = "./checkpoints_v2"
LAST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt")
SAVE_EVERY_N_STEPS = 200

# Path(s) to the PREVIOUS run's checkpoint (pre-regularization: old weight
# decay, old cross-attn LR, no label smoothing/extra dropout, unbalanced
# dataset). If LAST_CHECKPOINT_PATH (this run's own checkpoint) doesn't exist
# yet, we warm-start the model weights from here and then train with fresh
# optimizer/scheduler state using the NEW regularized hyperparameters --
# deliberately not inheriting the old optimizer momentum or LR schedule,
# since those were tuned for a different (less regularized) training setup.
#
# Checked in order; first one that exists wins. The Kaggle input dataset
# mount (read-only, from the earlier notebook version) is listed first since
# that's confirmed to hold the actual weights; the working-directory path is
# kept as a fallback in case a session already has a local copy.
OLD_CHECKPOINT_PATHS = [
    "/kaggle/input/notebooks/neelpraj/notebooke1c610a08a/checkpoints/last_checkpoint.pt",
    "./checkpoints/last_checkpoint.pt",
]

# Kaggle wipes /kaggle/working between sessions, so THIS run's own progress
# (saved to CHECKPOINT_DIR mid-session) is gone once the session restarts --
# even if you manually re-attach it as a Kaggle input dataset for the next
# session, it won't be at the expected working-dir LAST_CHECKPOINT_PATH.
# List any such re-attached locations here; if found, the checkpoint is
# copied into LAST_CHECKPOINT_PATH before the normal resume check runs, so
# it's treated as a genuine full resume (correct epoch/global_step/
# optimizer/scheduler) instead of being routed through the weights-only
# warm-start path (which always resets to WARM_START_EPOCH_OFFSET/step 0).
REATTACHED_CONTINUATION_CHECKPOINT_PATHS = [
    "/kaggle/input/notebooks/neelpraj/notebooke1c610a08a/checkpoints_v2/last_checkpoint.pt",
    "/kaggle/input/datasets/neelpraj/iuyt77/last_checkpoint.pt",
]


# ---------------------------------------------------------------------------
# Local dataset paths (datasets already downloaded onto disk -- no more
# HuggingFace Hub calls). Update these if your Kaggle dataset mount points
# ever change.
# ---------------------------------------------------------------------------
LOCAL_DATASET_PATHS = {
    "sroie_2019_text_recognition": "/kaggle/input/datasets/neelpraj/newwww/SROIE_2019",
    "iam_line": "/kaggle/input/datasets/neelpraj/newwww/IAM-line",
    "iiit5k_ocr": "/kaggle/input/datasets/neelpraj/newwww/IIIT5K_OCR",
    "icdar2019_sroie": "/kaggle/input/datasets/neelpraj/newwww/ICDAR2019-SROIE",
    # Two distinct copies of MJSynth are used as separate sources:
    #  - "newworld" copy is capped at MJSYNTH_CAP examples.
    #  - "newwww" copy is loaded in full (no cap).
    "mjsynth_text_recognition_capped": "/kaggle/input/datasets/neelpraj/newworld/MJSynth_text_recognition",

}


class ResizeWithPad:

    def __init__(self, target_height=384, target_width=384, pad_color=255):
        self.target_height = target_height
        self.target_width = target_width
        self.pad_color = pad_color

    def __call__(self, image: Image.Image) -> Image.Image:
        image = image.convert("RGB")
        orig_w, orig_h = image.size

        scale = self.target_height / orig_h
        new_w = int(orig_w * scale)
        new_h = self.target_height

        if new_w > self.target_width:
            new_w = self.target_width
            scale2 = self.target_width / orig_w
            new_h = min(self.target_height, int(orig_h * scale2))

        resized = image.resize((max(new_w, 1), max(new_h, 1)), Image.BICUBIC)

        canvas = Image.new("RGB", (self.target_width, self.target_height),
                            color=(self.pad_color, self.pad_color, self.pad_color))
        paste_x = 0
        paste_y = (self.target_height - new_h) // 2
        canvas.paste(resized, (paste_x, paste_y))

        return canvas


class OCRAugment:

    def __init__(self, p=0.2, max_rotation=3, seed=None):
        self.p = p
        self.max_rotation = max_rotation
        if seed is not None:
            random.seed(seed)

    def __call__(self, image: Image.Image) -> Image.Image:
        if random.random() < self.p:
            image = self._rotate(image)
        if random.random() < self.p:
            image = self._perspective_jitter(image)
        if random.random() < self.p:
            image = self._brightness_contrast(image)
        if random.random() < self.p * 0.5:
            image = self._blur(image)
        if random.random() < self.p * 0.5:
            image = self._noise(image)
        if random.random() < self.p * 0.3:
            image = self._random_erase(image)
        return image

    def _rotate(self, image):
        angle = random.uniform(-self.max_rotation, self.max_rotation)
        return image.rotate(angle, resample=Image.BICUBIC, expand=False,
                             fillcolor=(255, 255, 255))

    def _perspective_jitter(self, image):
        w, h = image.size
        jitter = 0.02

        def j(val, span):
            return val + random.uniform(-jitter, jitter) * span

        src = [(0, 0), (w, 0), (w, h), (0, h)]
        dst = [
            (j(0, w), j(0, h)),
            (j(w, w), j(0, h)),
            (j(w, w), j(h, h)),
            (j(0, w), j(h, h)),
        ]
        coeffs = self._find_perspective_coeffs(dst, src)
        return image.transform((w, h), Image.PERSPECTIVE, coeffs,
                                resample=Image.BICUBIC, fillcolor=(255, 255, 255))

    @staticmethod
    def _find_perspective_coeffs(pa, pb):
        matrix = []
        for p1, p2 in zip(pa, pb):
            matrix.append([p2[0], p2[1], 1, 0, 0, 0, -p1[0] * p2[0], -p1[0] * p2[1]])
            matrix.append([0, 0, 0, p2[0], p2[1], 1, -p1[1] * p2[0], -p1[1] * p2[1]])
        A = np.array(matrix, dtype=np.float64)
        B = np.array(pa).reshape(8)
        res = np.linalg.solve(A, B)
        return np.array(res).reshape(8)

    def _brightness_contrast(self, image):
        brightness_factor = random.uniform(0.8, 1.2)
        contrast_factor = random.uniform(0.8, 1.2)
        image = ImageEnhance.Brightness(image).enhance(brightness_factor)
        image = ImageEnhance.Contrast(image).enhance(contrast_factor)
        return image

    def _blur(self, image):
        radius = random.uniform(0.3, 1.0)
        return image.filter(ImageFilter.GaussianBlur(radius=radius))

    def _noise(self, image):
        arr = np.array(image).astype(np.float32)
        noise = np.random.normal(0, random.uniform(2, 8), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(arr)

    def _random_erase(self, image):
        w, h = image.size
        arr = np.array(image)
        erase_w = random.randint(int(w * 0.03), int(w * 0.12))
        erase_h = random.randint(int(h * 0.05), int(h * 0.20))
        x0 = random.randint(0, max(w - erase_w, 1))
        y0 = random.randint(0, max(h - erase_h, 1))
        arr[y0:y0 + erase_h, x0:x0 + erase_w] = 255
        return Image.fromarray(arr)


def augment_p_for_epoch(epoch, total_epochs, p_start=AUGMENT_P_START, p_end=AUGMENT_P_END):
    """
    Linearly ramps the augmentation probability from p_start (epoch 0) to
    p_end (final epoch). Keeps augmentation gentle while the model is still
    learning basic structure, then leans on it harder for regularization
    later in training.
    """
    if total_epochs <= 1:
        return p_end
    frac = epoch / (total_epochs - 1)
    frac = min(max(frac, 0.0), 1.0)
    return p_start + frac * (p_end - p_start)


def is_distributed_env():
    return "LOCAL_RANK" in os.environ and int(os.environ.get("WORLD_SIZE", "1")) > 1


def setup_environment():
    if is_distributed_env():
        dist.init_process_group(backend="nccl")
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
        device = torch.device(f"cuda:{local_rank}")
        is_main = (local_rank == 0)
        print_ = print if is_main else (lambda *a, **k: None)
        print_(f" Multi-GPU (DDP) active | world_size={dist.get_world_size()}")
        return True, local_rank, device, is_main
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"  Single-device run on {device}")
        return False, 0, device, True


def cleanup_environment(use_ddp):
    if use_ddp:
        dist.destroy_process_group()


def reduce_mean(value, device, use_ddp):

    if not use_ddp:
        return value
    t = torch.tensor(value).to(device)
    dist.all_reduce(t, op=dist.ReduceOp.SUM)
    return t.item() / dist.get_world_size()


def gather_all(obj_list, use_ddp):

    if not use_ddp:
        return [obj_list]
    world_size = dist.get_world_size()
    gathered = [None] * world_size
    dist.all_gather_object(gathered, obj_list)
    return gathered


def barrier(use_ddp, device_ids=None):
    if use_ddp:
        if device_ids:
            dist.barrier(device_ids=device_ids)
        else:
            dist.barrier()


def world_size(use_ddp):
    return dist.get_world_size() if use_ddp else 1


def strip_module_prefix(state_dict):

    return {(k[len("module."):] if k.startswith("module.") else k): v
            for k, v in state_dict.items()}


def shift_tokens_right(input_ids, pad_token_id, decoder_start_token_id):
    shifted = input_ids.new_zeros(input_ids.shape)
    shifted[:, 1:] = input_ids[:, :-1].clone()
    shifted[:, 0] = decoder_start_token_id
    return shifted


def compute_label_smoothed_loss(logits, labels, label_smoothing=LABEL_SMOOTHING):
    """
    Manual label-smoothed cross-entropy, since VisionEncoderDecoderModel's
    built-in `outputs.loss` doesn't expose a label_smoothing knob. `labels`
    uses -100 for padding, matching HF's ignore_index convention.
    """
    vocab_size = logits.size(-1)
    return torch.nn.functional.cross_entropy(
        logits.reshape(-1, vocab_size),
        labels.reshape(-1),
        ignore_index=-100,
        label_smoothing=label_smoothing,
    )


def apply_dropout_p(model, p):
    """
    Forces every nn.Dropout submodule in the model to use probability `p`.
    Changing model.config.*dropout* after from_pretrained() has no effect on
    already-constructed nn.Dropout modules (they capture `p` at construction
    time), so we walk the module tree directly instead.
    """
    n_updated = 0
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = p
            n_updated += 1
    return n_updated


def _cap(ds, n):
    if n is None or len(ds) <= n:
        return ds
    return ds.select(range(n))


# ---------------------------------------------------------------------------
# Local dataset loading helpers.
#
# These folders are plain local clones of the HF dataset repos (not
# `save_to_disk` exports), so we load them directly with the built-in
# "parquet" / "imagefolder" dataset builders. Both builders only touch local
# files -- no network / Hub calls are made.
# ---------------------------------------------------------------------------
def _load_parquet_split(pattern, is_main_process, name):
    """
    Loads a split from one or more local parquet shards matching `pattern`
    (a glob pattern, e.g. ".../data/train-*.parquet").
    """
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No parquet files matched pattern: {pattern}")
    if is_main_process:
        print(f"    Found {len(files)} parquet file(s) for {name}")
    return load_dataset("parquet", data_files=files, split="train")


_TEXT_COLUMN_CANDIDATES = ("text", "label", "ground_truth", "transcription", "words")


def _detect_text_column(ds, preferred=None):
    """
    Picks the most likely text/label column out of a dataset. If `preferred`
    is given and present, it wins; otherwise falls back to a fixed list of
    common column names used across OCR datasets.
    """
    if preferred and preferred in ds.column_names:
        return preferred
    for candidate in _TEXT_COLUMN_CANDIDATES:
        if candidate in ds.column_names:
            return candidate
    return None


def _fast_image_text_pairs(ds, is_main_process, desc, text_fn=None, text_column="text"):

    if "image" in ds.column_names:
        try:
            ds = ds.cast_column("image", HFImage(decode=False))
        except Exception:
            pass  # already non-decoding, or not an Image-typed column -- fine either way

    images = ds["image"]
    if text_fn is None:
        texts = ds[text_column]
        pairs = zip(images, texts)
    else:
        cols = ds[:]
        n = len(images)
        pairs = ((images[i], text_fn({k: v[i] for k, v in cols.items()})) for i in range(n))

    examples = []
    for image_ref, text in tqdm(pairs, total=len(images), desc=desc, disable=not is_main_process):
        if text is None or not str(text).strip():
            continue
        examples.append({"image": image_ref, "text": str(text)})
    return examples


def _normalize_sroie_text_recognition(is_main_process, max_samples=MAX_SAMPLES_PER_DATASET):
    # This folder is a raw image tree (train/train/*.jpg, test/test/*.jpg),
    # not parquet shards, so we use the local "imagefolder" builder, which
    # auto-detects train/test splits from the top-level folder names and
    # pairs images with any metadata.csv/metadata.jsonl it finds alongside
    # them. No network access is involved -- this is purely local.
    path = LOCAL_DATASET_PATHS["sroie_2019_text_recognition"]
    try:
        ds = load_dataset("imagefolder", data_dir=path, split="train")
        text_col = _detect_text_column(ds)
        if text_col is None:
            raise ValueError(
                "No recognizable text/label column found (looked for "
                f"{_TEXT_COLUMN_CANDIDATES}). Columns present: {ds.column_names}. "
                "This dataset folder may be missing its metadata.csv/metadata.jsonl "
                "label file, or it uses a differently named text column."
            )
        ds = _cap(ds, max_samples)
        return _fast_image_text_pairs(
            ds, is_main_process, desc="Normalizing SROIE_2019_text_recognition", text_column=text_col
        )
    except Exception as e:
        if is_main_process:
            print(f"Skipping SROIE_2019_text_recognition ({path}): {e}")
        return []


def _normalize_iam_line(is_main_process, max_samples=MAX_SAMPLES_PER_DATASET):
    # data/train.parquet, data/validation.parquet, data/test.parquet
    path = LOCAL_DATASET_PATHS["iam_line"]
    pattern = os.path.join(path, "data", "train.parquet")
    try:
        ds = _load_parquet_split(pattern, is_main_process, "IAM-line[train]")
        text_col = _detect_text_column(ds, preferred="text")
        if text_col is None:
            raise ValueError(f"No recognizable text column. Columns present: {ds.column_names}")
        ds = _cap(ds, max_samples)
        return _fast_image_text_pairs(ds, is_main_process, desc="Normalizing IAM-line", text_column=text_col)
    except Exception as e:
        if is_main_process:
            print(f" Skipping IAM-line ({path}): {e}")
        return []


def _normalize_iiit5k(is_main_process, max_samples=MAX_SAMPLES_PER_DATASET):
    # data/train-00000-of-00001.parquet, data/test-00000-of-00001.parquet
    # (there are also train_numbers-*/test_numbers-* shards -- a separate
    # digit-only subset -- which the "train-"/"test-" glob prefix naturally
    # excludes, since those filenames start with "train_"/"test_" not
    # "train-"/"test-")
    path = LOCAL_DATASET_PATHS["iiit5k_ocr"]
    examples = []
    for split in ("train", "test"):
        pattern = os.path.join(path, "data", f"{split}-*.parquet")
        try:
            ds = _load_parquet_split(pattern, is_main_process, f"IIIT5K_OCR[{split}]")
            text_col = _detect_text_column(ds)
            if text_col is None:
                raise ValueError(f"No recognizable text column. Columns present: {ds.column_names}")
            ds = _cap(ds, max_samples)
            examples.extend(_fast_image_text_pairs(
                ds, is_main_process, desc=f"Normalizing IIIT5K_OCR[{split}]", text_column=text_col
            ))
        except Exception as e:
            if is_main_process:
                print(f"Skipping IIIT5K_OCR[{split}] ({path}): {e}")
    return examples


def _normalize_icdar_sroie(is_main_process, max_samples=MAX_SAMPLES_PER_DATASET):
    def words_to_text(item):
        words = item.get("words")
        if not words:
            return None
        return " ".join(str(w) for w in words).strip()

    # data/train-00000-of-00001.parquet, data/test-00000-of-00001.parquet
    path = LOCAL_DATASET_PATHS["icdar2019_sroie"]
    examples = []
    for split in ("train", "test"):
        pattern = os.path.join(path, "data", f"{split}-*.parquet")
        try:
            ds = _load_parquet_split(pattern, is_main_process, f"ICDAR2019-SROIE[{split}]")
            if "words" in ds.column_names:
                ds = _cap(ds, max_samples)
                examples.extend(_fast_image_text_pairs(
                    ds, is_main_process, desc=f"Normalizing ICDAR2019-SROIE[{split}]", text_fn=words_to_text
                ))
            else:
                text_col = _detect_text_column(ds)
                if text_col is None:
                    raise ValueError(f"No 'words' or recognizable text column. Columns present: {ds.column_names}")
                ds = _cap(ds, max_samples)
                examples.extend(_fast_image_text_pairs(
                    ds, is_main_process, desc=f"Normalizing ICDAR2019-SROIE[{split}]", text_column=text_col
                ))
        except Exception as e:
            if is_main_process:
                print(f"  Skipping ICDAR2019-SROIE[{split}] ({path}): {e}")
    return examples


def _normalize_mjsynth(is_main_process, path, max_samples, label):
    # data/train-00000-of-00025-<hash>.parquet ... N shards total.
    pattern = os.path.join(path, "data", "train-*.parquet")
    try:
        ds = _load_parquet_split(pattern, is_main_process, f"{label}[train]")
        text_col = _detect_text_column(ds, preferred="label")
        if text_col is None:
            raise ValueError(f"No recognizable text column. Columns present: {ds.column_names}")
        ds = _cap(ds, max_samples)
        return _fast_image_text_pairs(
            ds, is_main_process, desc=f"Normalizing {label}", text_column=text_col
        )
    except Exception as e:
        if is_main_process:
            print(f" Skipping {label} ({path}): {e}")
        return []


def _normalize_mjsynth_capped(is_main_process, max_samples=None):
    # "newworld" copy -- capped at MJSYNTH_CAP examples.
    return _normalize_mjsynth(
        is_main_process,
        path=LOCAL_DATASET_PATHS["mjsynth_text_recognition_capped"],
        max_samples=MJSYNTH_CAP,
        label="MJSynth_text_recognition (capped, newworld)",
    )


def load_combined_examples(is_main_process):
    loaders = [
        ("SROIE_2019_text_recognition", _normalize_sroie_text_recognition),
        ("IAM-line", _normalize_iam_line),
        ("IIIT5K_OCR", _normalize_iiit5k),
        ("ICDAR2019-SROIE", _normalize_icdar_sroie),
        ("MJSynth_text_recognition (capped, newworld)", _normalize_mjsynth_capped),

    ]
    all_examples = []
    for name, fn in loaders:
        if is_main_process:
            print(f"⏳ Loading {name} (local) ...")
        examples = fn(is_main_process)
        raw_count = len(examples)

        # --- Dataset rebalancing: oversample under-represented real-world
        # sources so MJSynth (clean, synthetic) doesn't dominate the epoch
        # and drive the model toward memorizing synthetic rendering patterns
        # instead of learning transferable visual-to-text features. ---
        factor = OVERSAMPLE_FACTORS.get(name, 1)
        if factor > 1:
            examples = examples * factor

        if is_main_process:
            if factor > 1:
                print(f"   -> {raw_count} usable examples (oversampled x{factor} -> {len(examples)})")
            else:
                print(f"   -> {len(examples)} usable examples")
        all_examples.extend(examples)
    return all_examples


def split_examples(examples, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED):

    rng = random.Random(seed)
    shuffled = examples[:]
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_val = int(n * val_fraction)
    n_test = int(n * test_fraction)
    val = shuffled[:n_val]
    test = shuffled[n_val:n_val + n_test]
    train = shuffled[n_val + n_test:]
    return train, val, test


class CombinedOCRDataset(Dataset):
    def __init__(self, raw_data, processor, tokenizer, max_length=128,
                 is_train=True, img_height=384, img_width=384, augment_p=AUGMENT_P_START):
        self.raw_data = raw_data
        self.processor = processor
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_train = is_train
        self.resize_pad = ResizeWithPad(target_height=img_height, target_width=img_width)
        self.augment = OCRAugment(p=augment_p) if is_train else None

    def set_augment_p(self, p):
        """Called once per epoch from the training loop to ramp augmentation strength."""
        if self.augment is not None:
            self.augment.p = p

    def __len__(self):
        return len(self.raw_data)

    def __getitem__(self, idx):
        item = self.raw_data[idx]
        image = self._decode_image(item["image"])
        image = image.convert("RGB")
        label_text = item["text"]

        if self.is_train and self.augment is not None:
            image = self.augment(image)

        image = self.resize_pad(image)

        pixel_values = self.processor(
            images=image, return_tensors="pt", do_resize=False
        ).pixel_values.squeeze(0)

        labels = self.tokenizer(
            label_text,
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            add_special_tokens=True
        ).input_ids

        labels = [label if label != self.tokenizer.pad_token_id else -100 for label in labels]
        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(labels, dtype=torch.long)
        }

    @staticmethod
    def _decode_image(image_ref):

        if isinstance(image_ref, Image.Image):
            return image_ref
        if isinstance(image_ref, dict):
            image_bytes = image_ref.get("bytes")
            if image_bytes is not None:
                return Image.open(io.BytesIO(image_bytes))
            path = image_ref.get("path")
            if path:
                return CombinedOCRDataset._open_path(path)
            raise ValueError(f"Unrecognized image dict shape: {list(image_ref.keys())}")
        if isinstance(image_ref, (str, bytes)):
            return CombinedOCRDataset._open_path(image_ref)
        raise TypeError(f"Unrecognized image reference type: {type(image_ref)}")

    @staticmethod
    def _open_path(path):
        if isinstance(path, str) and "://" in path:

            with fsspec.open(path, "rb") as f:
                return Image.open(io.BytesIO(f.read()))
        return Image.open(path)


def evaluate_loss(model, loader, tokenizer, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            labels_for_shift = labels.clone()
            labels_for_shift[labels_for_shift == -100] = tokenizer.pad_token_id
            decoder_input_ids = shift_tokens_right(
                labels_for_shift, tokenizer.pad_token_id, tokenizer.eos_token_id
            )

            with torch.amp.autocast('cuda'):
                outputs = model(
                    pixel_values=pixel_values,
                    decoder_input_ids=decoder_input_ids,
                )
                loss = compute_label_smoothed_loss(outputs.logits, labels)

            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)


def run_test_metrics(model, test_loader, tokenizer, device, is_main_process, use_ddp):
    model.eval()
    local_predictions = []
    local_ground_truths = []
    total_inference_time = 0.0
    total_images_processed = 0

    raw_model = model.module if use_ddp else model

    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"]
            num_images_in_batch = pixel_values.size(0)

            start_time = time.perf_counter()
            with torch.amp.autocast('cuda'):
                generated_ids = raw_model.generate(
                    pixel_values,
                    max_new_tokens=128,
                    early_stopping=True,
                    num_beams=2,
                    do_sample=False,
                    repetition_penalty=1.2,
                    no_repeat_ngram_size=2,
                )
            end_time = time.perf_counter()

            total_inference_time += (end_time - start_time)
            total_images_processed += num_images_in_batch

            predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

            labels[labels == -100] = tokenizer.pad_token_id
            ground_truths = tokenizer.batch_decode(labels, skip_special_tokens=True)

            for pred, gt in zip(predictions, ground_truths):
                local_predictions.append(pred.strip())
                local_ground_truths.append(gt.strip())

    global_predictions = gather_all(local_predictions, use_ddp)
    global_ground_truths = gather_all(local_ground_truths, use_ddp)

    if is_main_process:
        all_predictions = [item for sublist in global_predictions for item in sublist]
        all_ground_truths = [item for sublist in global_ground_truths for item in sublist]

        for gt, pred in zip(all_ground_truths[:100], all_predictions[:100]):
            print(f" Target: {gt:<12} | Prediction: {pred}")

        total_cer = cer(all_ground_truths, all_predictions)
        total_wer = wer(all_ground_truths, all_predictions)

        exact_matches = sum(1 for gt, pred in zip(all_ground_truths, all_predictions) if gt == pred)
        em_accuracy = (exact_matches / len(all_ground_truths)) * 100 if all_ground_truths else 0.0

        avg_latency_per_img = (total_inference_time / total_images_processed) * 1000 if total_images_processed else 0.0
        throughput_per_sec = (total_images_processed * world_size(use_ddp)) / total_inference_time if total_inference_time else 0.0

        print("\n" + "=" * 50)
        print("FINAL PRODUCTION METRICS SUMMARY (TEST SPLIT)")
        print("=" * 50)
        print(f"1. Character Error Rate (CER) : {total_cer * 100:.2f}% (Accuracy: {(1 - total_cer) * 100:.2f}%)")
        print(f"2. Word Error Rate (WER)      : {total_wer * 100:.2f}%")
        print(f"3. Exact Match Accuracy       : {em_accuracy:.2f}%")
        print(f"4. Avg Latency Per Device     : {avg_latency_per_img:.2f} ms/image")
        print(f"5. Total Throughput Fleet     : {throughput_per_sec:.2f} images/sec")
        print("=" * 50)


def config_snapshot():
    return {
        "SEED": SEED,
        "MAX_SAMPLES_PER_DATASET": MAX_SAMPLES_PER_DATASET,
        "MJSYNTH_CAP": MJSYNTH_CAP,
        "VAL_FRACTION": VAL_FRACTION,
        "TEST_FRACTION": TEST_FRACTION,
        "OVERSAMPLE_FACTORS": OVERSAMPLE_FACTORS,
    }


def build_optimizer(model):
    """
    Builds an AdamW optimizer with four param groups:
      - pretrained params, with weight decay
      - pretrained params, no weight decay (biases / LayerNorm)
      - cross-attention params (randomly initialized), with weight decay, higher LR
      - cross-attention params, no weight decay, higher LR

    Cross-attention is the only part of a Swin-encoder / BART-decoder
    VisionEncoderDecoderModel that has no pretrained initialization (the
    encoder and decoder self-attention blocks are pretrained; the
    encoder<->decoder cross-attention is created fresh by
    VisionEncoderDecoderModel). Giving it a higher LR lets it catch up
    without needing to blow up the LR for the whole model.
    """
    no_decay_keywords = ("bias", "LayerNorm.weight", "layer_norm.weight")

    cross_attn_decay, cross_attn_no_decay = [], []
    other_decay, other_no_decay = [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        is_cross_attn = "encoder_attn" in name
        is_no_decay = any(k in name for k in no_decay_keywords)

        if is_cross_attn and is_no_decay:
            cross_attn_no_decay.append(param)
        elif is_cross_attn:
            cross_attn_decay.append(param)
        elif is_no_decay:
            other_no_decay.append(param)
        else:
            other_decay.append(param)

    param_groups = [
        {"params": other_decay, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": other_no_decay, "lr": LEARNING_RATE, "weight_decay": 0.0},
        {"params": cross_attn_decay, "lr": CROSS_ATTN_LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": cross_attn_no_decay, "lr": CROSS_ATTN_LEARNING_RATE, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(param_groups)


def save_checkpoint(path, raw_model, optimizer, scaler, scheduler, epoch, epoch_complete,
                     global_step, best_val_loss, is_main_process):

    if not is_main_process:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp_path = path + ".tmp"
    torch.save({
        "model_state_dict": raw_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "epoch": epoch,
        "epoch_complete": epoch_complete,
        "global_step": global_step,
        "best_val_loss": best_val_loss,
        "config_snapshot": config_snapshot(),
    }, tmp_path)
    os.replace(tmp_path, path)  # atomic-ish: avoids a half-written file if interrupted mid-save
    print(f"💾 Checkpoint saved -> {path} (epoch {epoch}, step {global_step})")


def _resolve_checkpoint_file(path):
    """
    Resolves a candidate checkpoint path to an actual file. Handles the
    common Kaggle-dataset-upload case where uploading a folder results in
    the .pt file being nested one level deeper than expected (e.g. a folder
    literally named "last_checkpoint.pt" containing the real file inside
    it), by searching for a *.pt file if `path` turns out to be a directory.
    """
    if os.path.isfile(path):
        return path
    if os.path.isdir(path):
        # Prefer an exact "last_checkpoint.pt" match if present, otherwise
        # take the first *.pt file found anywhere under this directory.
        exact = os.path.join(path, "last_checkpoint.pt")
        if os.path.isfile(exact):
            return exact
        matches = glob.glob(os.path.join(path, "**", "*.pt"), recursive=True)
        if matches:
            return matches[0]
    return None


def recover_reattached_continuation_checkpoint(candidate_paths, local_target_path, is_main_process):
    """
    Checks `candidate_paths` for a continuation-phase checkpoint that got
    re-attached as a Kaggle input after a session reset wiped /kaggle/working.
    If one is found (resolving through directories the same way
    _resolve_checkpoint_file does), it's copied to `local_target_path` so the
    normal load_checkpoint_if_exists() resume path picks it up -- preserving
    the real epoch/global_step/optimizer/scheduler state instead of treating
    it as a from-scratch warm-start source.

    If multiple candidates resolve to real files, the one with the highest
    saved `global_step` wins (most training progress).
    """
    resolved_candidates = []
    for path in candidate_paths:
        resolved = _resolve_checkpoint_file(path)
        if resolved is not None:
            resolved_candidates.append(resolved)

    if not resolved_candidates:
        return False

    best_path, best_step = None, -1
    for path in resolved_candidates:
        try:
            ckpt = torch.load(path, map_location="cpu")
            step = ckpt.get("global_step", 0)
        except Exception as e:
            if is_main_process:
                print(f"    Could not inspect candidate continuation checkpoint {path}: {e}")
            continue
        if step > best_step:
            best_path, best_step = path, step

    if best_path is None:
        return False

    os.makedirs(os.path.dirname(local_target_path), exist_ok=True)
    shutil.copy2(best_path, local_target_path)
    if is_main_process:
        print(f"♻️  Recovered re-attached continuation checkpoint from {best_path} "
              f"(global_step={best_step}) -> {local_target_path}")
    return True


def load_warm_start_weights(paths, raw_model, device, is_main_process):
    """
    Loads ONLY the model weights from a previous run's checkpoint -- no
    optimizer, scheduler, or scaler state, and no epoch/global_step bookkeeping.
    Used when switching to a meaningfully different training regime (new
    weight decay, new cross-attn LR, label smoothing, extra dropout,
    rebalanced dataset) where inheriting the old optimizer momentum or LR
    schedule position would fight against the new settings rather than help.

    `paths` is a list of candidate locations, checked in order; the first
    one that resolves to an actual file is used (each candidate may itself
    be a directory, e.g. a Kaggle dataset upload that nested the .pt file --
    see _resolve_checkpoint_file). If NONE resolve, this raises rather than
    silently falling through to a randomly-initialized model -- warm-starting
    was explicitly requested, so a missing checkpoint should stop the run,
    not quietly train from scratch.
    """
    for path in paths:
        resolved = _resolve_checkpoint_file(path)
        if resolved is not None:
            if is_main_process:
                if resolved != path:
                    print(f"    (candidate path {path} is a directory -- resolved to {resolved})")
                print(f"🔥 Warm-starting model weights from previous run's checkpoint: {resolved}")
            ckpt = torch.load(resolved, map_location=device)
            state_dict = strip_module_prefix(ckpt["model_state_dict"])
            raw_model.load_state_dict(state_dict)
            if is_main_process:
                prev_val_loss = ckpt.get("best_val_loss")
                if prev_val_loss is not None:
                    print(f"    (previous run's best_val_loss was {prev_val_loss:.4f} -- "
                          "optimizer/scheduler are starting fresh with the new regularized settings)")
            return resolved

    raise FileNotFoundError(
        "Warm-start was requested but none of the candidate checkpoint paths resolved to an "
        f"actual file: {paths}. Refusing to silently fall back to a randomly-initialized model -- "
        "check that the expected Kaggle input dataset is attached to this session, that the "
        "path isn't a directory with no .pt file inside it, or update OLD_CHECKPOINT_PATHS."
    )


def load_checkpoint_if_exists(path, raw_model, optimizer, scaler, scheduler, device, is_main_process):
    """
    Returns (start_epoch, global_step, best_val_loss, scheduler_restored).
    `scheduler_restored` tells the caller whether the scheduler's internal
    step counter actually matches `global_step` -- if it doesn't (fresh
    scheduler after a param-group-incompatible checkpoint), the caller should
    fast-forward it manually so the cosine curve doesn't restart from zero
    while global_step has already advanced.
    """

    if not os.path.exists(path):
        if is_main_process:
            print("  No checkpoint found -- starting fresh.")
        return 0, 0, float("inf"), True

    if is_main_process:
        print(f" Found checkpoint at {path} -- resuming.")
    ckpt = torch.load(path, map_location=device)

    state_dict = strip_module_prefix(ckpt["model_state_dict"])
    raw_model.load_state_dict(state_dict)

    # Optimizer/scheduler state is tied to the exact param-group structure that
    # produced it. A checkpoint saved before the differential-LR change (single
    # param group) is incompatible with the current 4-group optimizer, and the
    # scheduler's LambdaLR state is likewise per-group. Rather than crashing,
    # fall back to fresh optimizer/scheduler state in that case -- model
    # weights and training progress (epoch/global_step/best_val_loss) still
    # resume correctly, only the optimizer momentum and LR-schedule position
    # are reset (and then fast-forwarded back into sync by the caller).
    try:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    except ValueError as e:
        if is_main_process:
            print(f"    Could not restore optimizer state ({e}). This is expected if this "
                  "checkpoint predates the differential-LR/weight-decay change. "
                  "Continuing with a freshly initialized optimizer.")

    scheduler_restored = False
    if "scheduler_state_dict" in ckpt:
        try:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            scheduler_restored = True
        except (ValueError, KeyError, RuntimeError) as e:
            if is_main_process:
                print(f"    Could not restore scheduler state ({e}). "
                      "Continuing with a freshly initialized LR schedule (will fast-forward).")
    elif is_main_process:
        print("    No scheduler state in checkpoint (older run) -- scheduler will be "
              "fast-forwarded to match global_step.")

    try:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    except (ValueError, KeyError, RuntimeError) as e:
        if is_main_process:
            print(f"    Could not restore AMP scaler state ({e}). Continuing with a fresh scaler.")

    saved_cfg = ckpt.get("config_snapshot", {})
    current_cfg = config_snapshot()
    if is_main_process and saved_cfg and saved_cfg != current_cfg:
        print("  WARNING: data config differs from the checkpoint's saved config!")
        print(f"    Saved:   {saved_cfg}")
        print(f"    Current: {current_cfg}")
        print("    Your train/val/test split may no longer match the one used to "
              "train this checkpoint. Fix SEED/caps/fractions to match, or accept "
              "the drift knowingly.")

    global_step = ckpt.get("global_step", 0)
    best_val_loss = ckpt.get("best_val_loss", float("inf"))

    if ckpt.get("epoch_complete", True):
        start_epoch = ckpt["epoch"] + 1
    else:
        start_epoch = ckpt["epoch"]
        if is_main_process:
            print(f"    Last epoch ({ckpt['epoch']}) was interrupted mid-run -- redoing it.")

    if is_main_process:
        print(f"    Resuming at epoch {start_epoch}, global_step {global_step}, "
              f"best_val_loss {best_val_loss:.4f}")
    return start_epoch, global_step, best_val_loss, scheduler_restored


def main():
    use_ddp, local_rank, device, is_main_process = setup_environment()

    processor = AutoImageProcessor.from_pretrained("microsoft/swin-base-patch4-window12-384")
    tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

    all_examples = load_combined_examples(is_main_process)
    if is_main_process:
        print(f"📦 Total combined examples across all sources: {len(all_examples)}")

    train_raw, val_raw, test_raw = split_examples(all_examples)
    if is_main_process:
        print(f"   Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}")

    train_dataset = CombinedOCRDataset(train_raw, processor, tokenizer, is_train=True,
                                        augment_p=AUGMENT_P_START)
    val_dataset = CombinedOCRDataset(val_raw, processor, tokenizer, is_train=False)
    test_dataset = CombinedOCRDataset(test_raw, processor, tokenizer, is_train=False)

    if use_ddp:
        train_sampler = DistributedSampler(train_dataset, shuffle=True)
        val_sampler = DistributedSampler(val_dataset, shuffle=False)
        test_sampler = DistributedSampler(test_dataset, shuffle=False)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
                                   num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, sampler=val_sampler,
                                 num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, sampler=test_sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    else:
        train_sampler = None
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                   num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)

    encoder = SwinModel.from_pretrained("microsoft/swin-base-patch4-window12-384")
    decoder = BartForCausalLM.from_pretrained("facebook/bart-base")
    model = VisionEncoderDecoderModel(encoder=encoder, decoder=decoder).to(device)

    model.config.encoder.add_cross_attention = False
    model.config.decoder.add_cross_attention = True
    model.config.decoder.is_decoder = True

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.decoder_start_token_id = tokenizer.eos_token_id
    model.config.vocab_size = model.config.decoder.vocab_size

    # --- Increased dropout for regularization (applied directly to the
    # already-constructed nn.Dropout modules; see apply_dropout_p) ---
    n_dropout_layers = apply_dropout_p(model, DROPOUT_P)
    if is_main_process:
        print(f"    Set dropout p={DROPOUT_P} on {n_dropout_layers} Dropout layers")

    generation_config = GenerationConfig.from_model_config(model.config)
    generation_config.decoder_start_token_id = tokenizer.eos_token_id
    generation_config.pad_token_id = tokenizer.pad_token_id
    generation_config.eos_token_id = tokenizer.eos_token_id
    generation_config.vocab_size = model.config.decoder.vocab_size
    generation_config.forced_eos_token_id = tokenizer.eos_token_id
    model.generation_config = generation_config

    if use_ddp:
        model = DDP(model, device_ids=[local_rank], output_device=local_rank)

    raw_model = model.module if use_ddp else model
    total_params = sum(p.numel() for p in model.parameters())

    # --- differential LR (pretrained vs. fresh cross-attention) ---
    # --- explicit weight decay, excluding bias/LayerNorm params ---
    optimizer = build_optimizer(raw_model)
    scaler = torch.amp.GradScaler('cuda')

    # --- cosine LR schedule with linear warmup ---
    # num_update_steps counts actual optimizer.step() calls (i.e. after
    # gradient accumulation), not raw micro-batches, since the scheduler
    # should advance once per real parameter update.
    #
    # Sized to the epochs THIS phase will actually run (EPOCHS -
    # WARM_START_EPOCH_OFFSET = 10), not the absolute EPOCHS total (30) --
    # this schedule starts fresh at global_step=0 here, so if it were sized
    # for 30 epochs it would only complete 1/3 of its own cosine curve by the
    # time training actually ends, leaving the final LR much higher than
    # intended (the same class of bug as the earlier checkpoint-resume case).
    schedule_epochs = EPOCHS - WARM_START_EPOCH_OFFSET
    assert schedule_epochs > 0, "EPOCHS must be greater than WARM_START_EPOCH_OFFSET"
    num_micro_batches_per_epoch = len(train_loader)
    num_update_steps_per_epoch = math.ceil(num_micro_batches_per_epoch / GRAD_ACCUM_STEPS)
    num_training_steps = num_update_steps_per_epoch * schedule_epochs
    num_warmup_steps = max(1, int(WARMUP_RATIO * num_training_steps))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
    )

    if not os.path.exists(LAST_CHECKPOINT_PATH):
        # This session's local working directory has no checkpoint of its own
        # (expected right after a Kaggle session restart, since /kaggle/working
        # is wiped). Before falling back to the pre-regularization warm-start,
        # check whether a genuine continuation-phase checkpoint was manually
        # re-attached as an input dataset -- if so, recover it into place so
        # it resumes with its real epoch/global_step/optimizer/scheduler state.
        recover_reattached_continuation_checkpoint(
            REATTACHED_CONTINUATION_CHECKPOINT_PATHS, LAST_CHECKPOINT_PATH, is_main_process
        )

    if os.path.exists(LAST_CHECKPOINT_PATH):
        # This regularized run already has its own checkpoint (e.g. it was
        # interrupted partway through) -- do a normal full resume, including
        # optimizer/scheduler/scaler state, since that state was produced
        # under the current (new) hyperparameters and is safe to restore.
        start_epoch, global_step, best_val_loss, scheduler_restored = load_checkpoint_if_exists(
            LAST_CHECKPOINT_PATH, raw_model, optimizer, scaler, scheduler, device, is_main_process
        )
        # If the scheduler couldn't be restored (e.g. a param-group mismatch),
        # its internal step counter is at 0 even though `global_step` already
        # reflects real progress. Fast-forward it so the cosine curve picks up
        # from the correct position instead of restarting the anneal.
        if not scheduler_restored and global_step > 0:
            if is_main_process:
                print(f"    Fast-forwarding LR schedule by {global_step} steps to match global_step...")
            for _ in range(global_step):
                scheduler.step()
    else:
        # No checkpoint for this run yet -- warm-start the model weights from
        # the previous (pre-regularization) run, and start this continuation
        # phase with completely fresh optimizer/scheduler/scaler state built
        # from the NEW regularized hyperparameters (weight decay, cross-attn
        # LR, etc.). This deliberately does NOT inherit old optimizer momentum
        # or LR-schedule position, since those were tuned for a different
        # (less regularized) training setup and dataset mixture.
        load_warm_start_weights(OLD_CHECKPOINT_PATHS, raw_model, device, is_main_process)
        start_epoch, global_step, best_val_loss = WARM_START_EPOCH_OFFSET, 0, float("inf")

    if is_main_process:
        print("Starting/resuming training pipeline...")
        print(f"    LR schedule: {num_warmup_steps} warmup steps / {num_training_steps} total steps")

    for epoch in range(start_epoch, EPOCHS):
        if use_ddp:
            train_sampler.set_epoch(epoch)

        # --- ramp augmentation strength across epochs ---
        # Uses the LOCAL epoch index within this continuation phase (0..9),
        # not the absolute display epoch (20..29) -- same reasoning as
        # schedule_epochs above: the ramp should span this phase's actual
        # 10 epochs, not get pinned near the ceiling because the absolute
        # epoch number is already high relative to EPOCHS=30.
        local_epoch = epoch - WARM_START_EPOCH_OFFSET
        current_augment_p = augment_p_for_epoch(local_epoch, schedule_epochs)
        train_dataset.set_augment_p(current_augment_p)
        if is_main_process:
            print(f"    Augmentation probability for epoch {epoch+1}: {current_augment_p:.3f}")

        model.train()
        total_loss = 0.0
        n_batches_this_epoch = 0

        epoch_iter = tqdm(train_loader, desc=f"Continuation epoch {epoch+1}/{EPOCHS} [train]", disable=not is_main_process)
        optimizer.zero_grad()
        for micro_step, batch in enumerate(epoch_iter):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            labels_for_shift = labels.clone()
            labels_for_shift[labels_for_shift == -100] = tokenizer.pad_token_id
            decoder_input_ids = shift_tokens_right(
                labels_for_shift, tokenizer.pad_token_id, tokenizer.eos_token_id
            )

            with torch.amp.autocast('cuda'):
                outputs = model(
                    pixel_values=pixel_values,
                    decoder_input_ids=decoder_input_ids,
                )
                # --- label-smoothed loss instead of the model's built-in
                # (unsmoothed) loss, to shrink the train/val gap ---
                loss = compute_label_smoothed_loss(outputs.logits, labels)
                # Scale down before backward so accumulated gradients across
                # GRAD_ACCUM_STEPS micro-batches sum to the same magnitude as
                # a single full-size batch would have produced.
                loss_to_backward = loss / GRAD_ACCUM_STEPS
            scaler.scale(loss_to_backward).backward()

            total_loss += loss.item()  # unscaled, for reporting
            n_batches_this_epoch += 1

            is_last_micro_batch = (micro_step + 1) == num_micro_batches_per_epoch
            if (micro_step + 1) % GRAD_ACCUM_STEPS == 0 or is_last_micro_batch:
                # --- gradient clipping (must unscale before clipping under AMP) ---
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

                global_step += 1
                if is_main_process:
                    current_lr = scheduler.get_last_lr()[0]
                    epoch_iter.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.2e}")

                if global_step % SAVE_EVERY_N_STEPS == 0:
                    barrier(use_ddp, device_ids=[local_rank] if use_ddp else None)
                    save_checkpoint(LAST_CHECKPOINT_PATH, raw_model, optimizer, scaler, scheduler,
                                     epoch=epoch, epoch_complete=False,
                                     global_step=global_step, best_val_loss=best_val_loss,
                                     is_main_process=is_main_process)
                    # Extra safeguard against GPU memory fragmentation building
                    # up over a long run -- cheap relative to a mid-epoch OOM crash.
                    torch.cuda.empty_cache()

        avg_loss = total_loss / max(n_batches_this_epoch, 1)
        global_avg_train_loss = reduce_mean(avg_loss, device, use_ddp)

        local_val_loss = evaluate_loss(model, val_loader, tokenizer, device)
        global_avg_val_loss = reduce_mean(local_val_loss, device, use_ddp)

        if is_main_process:
            print(f"Continuation epoch {epoch+1:03d}/{EPOCHS} | Train Loss: {global_avg_train_loss:.4f} | Val Loss: {global_avg_val_loss:.4f}")

        barrier(use_ddp, device_ids=[local_rank] if use_ddp else None)
        save_checkpoint(LAST_CHECKPOINT_PATH, raw_model, optimizer, scaler, scheduler,
                         epoch=epoch, epoch_complete=True,
                         global_step=global_step, best_val_loss=best_val_loss,
                         is_main_process=is_main_process)

        if is_main_process and global_avg_val_loss < best_val_loss:
            best_val_loss = global_avg_val_loss
            os.makedirs(BEST_MODEL_DIR, exist_ok=True)
            raw_model.save_pretrained(BEST_MODEL_DIR)
            processor.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            print(f" New best val loss ({best_val_loss:.4f}) -> saved to '{BEST_MODEL_DIR}'")

            save_checkpoint(LAST_CHECKPOINT_PATH, raw_model, optimizer, scaler, scheduler,
                             epoch=epoch, epoch_complete=True,
                             global_step=global_step, best_val_loss=best_val_loss,
                             is_main_process=is_main_process)

    barrier(use_ddp, device_ids=[local_rank] if use_ddp else None)

    if is_main_process:
        print(f"\n Training concluded. Exporting final weights to: '{FINAL_MODEL_DIR}'")
        os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
        raw_model.save_pretrained(FINAL_MODEL_DIR)
        processor.save_pretrained(FINAL_MODEL_DIR)
        tokenizer.save_pretrained(FINAL_MODEL_DIR)

    if is_main_process:
        print("\n Running Inference and Metric Benchmarking on TEST split...")
        print(f"6. Total Parameter Size       : {total_params / 1e6:.1f}M")

    run_test_metrics(model, test_loader, tokenizer, device, is_main_process, use_ddp)

    cleanup_environment(use_ddp)


if __name__ == "__main__":
    main()

Overwriting train_ocr.py


In [5]:
!torchrun --nproc_per_node=2 train_ocr.py

W0719 11:24:06.428000 115 torch/distributed/run.py:852] 
W0719 11:24:06.428000 115 torch/distributed/run.py:852] *****************************************
W0719 11:24:06.428000 115 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0719 11:24:06.428000 115 torch/distributed/run.py:852] *****************************************
 Multi-GPU (DDP) active | world_size=2
preprocessor_config.json: 100%|████████████████| 255/255 [00:00<00:00, 1.38MB/s]
The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
The image processor of type `ViTImageProcessor` is